# Phase 10 — Testing & Validation (Colab)
Run the complete voice analysis pipeline on 3 test videos using Colab GPU.

**Prerequisites:**
- GitHub repo: `anvay-cpu/voice-analysis-pipeline` (private)
- Google Drive:
  - `My Drive/best_model.pt` (disfluency model, 361MB)
  - `My Drive/test_videos/test_good_speaker_5min.mp4`
  - `My Drive/test_videos/test_nervous_speaker_5min.mp4`
  - `My Drive/test_videos/test_monotone_speaker_5min.mp4`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo

In [ ]:
!git clone https://github.com/anvay-cpu/voice-analysis-pipeline.git
%cd voice-analysis-pipeline

## 3. Install dependencies

In [ ]:
!pip install -q transformers librosa parselmouth-dummy praat-parselmouth speechbrain pyyaml psutil
# Fix SpeechBrain compatibility with newer torchaudio/huggingface_hub
!pip install -q huggingface_hub==0.23.0

## 4. Copy large files from Drive

In [ ]:
import os
import shutil

# Create directories
os.makedirs('models/disfluency', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/outputs', exist_ok=True)

# Copy disfluency model
src = '/content/drive/MyDrive/best_model.pt'
dst = 'models/disfluency/best_model.pt'
if not os.path.exists(dst):
    print(f'Copying disfluency model ({os.path.getsize(src)/1e6:.0f} MB)...')
    shutil.copy2(src, dst)
    print('Done.')
else:
    print('Disfluency model already in place.')

# Copy test videos
videos = [
    'test_good_speaker_5min.mp4',
    'test_nervous_speaker_5min.mp4',
    'test_monotone_speaker_5min.mp4',
]
for v in videos:
    src = f'/content/drive/MyDrive/test_videos/{v}'
    dst = f'data/raw/{v}'
    if not os.path.exists(dst):
        size = os.path.getsize(src) / 1e6
        print(f'Copying {v} ({size:.1f} MB)...')
        shutil.copy2(src, dst)
    else:
        print(f'{v} already in place.')

print('\nAll files ready!')
print(f'Disfluency model: {os.path.getsize("models/disfluency/best_model.pt")/1e6:.0f} MB')
for v in videos:
    print(f'{v}: {os.path.getsize(f"data/raw/{v}")/1e6:.1f} MB')

## 5. Verify all models can load

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Patch config to use CUDA instead of MPS
import yaml

with open('configs/pipeline_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
config['pipeline']['device'] = device
print(f'Using device: {device}')

with open('configs/pipeline_config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print('Config updated.')

In [ ]:
# Quick model load test
print('Testing Whisper...')
from src.transcriber import load_whisper
pipe = load_whisper(device=device)
print('  OK')

print('Testing Disfluency...')
from src.disfluency_model import load_disfluency_model
dis_model, dis_proc = load_disfluency_model('models/disfluency/best_model.pt', device=device)
print('  OK')

print('Testing Emotion...')
from src.vocal_emotion import load_emotion_model
emo = load_emotion_model('models/vocal_emotion/best_model.pt', device=device)
print('  OK')

print('\nAll models loaded successfully!')

# Clean up to free memory before pipeline run
del pipe, dis_model, dis_proc, emo
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 6. Run Phase 10 — Full Pipeline on 3 Videos

In [ ]:
!python -u run_phase10.py

## 7. Inspect Results

In [ ]:
import json
import os

output_dir = 'data/outputs'
for f in sorted(os.listdir(output_dir)):
    if f.endswith('.json'):
        path = os.path.join(output_dir, f)
        with open(path) as fp:
            data = json.load(fp)
        
        meta = data['metadata']
        summary = data['summary']
        
        print(f'\n{"="*60}')
        print(f'  {f}')
        print(f'{"="*60}')
        print(f'  Duration:      {meta["duration_sec"]:.0f}s')
        print(f'  Words:         {summary["word_count"]} ({summary["words_per_minute"]:.0f} WPM)')
        print(f'  Fillers:       {summary["filler_count"]} ({summary["fillers_per_minute"]:.1f}/min)')
        print(f'  Disfluencies:  {summary["disfluency_count"]}')
        if summary.get('disfluency_types'):
            print(f'    Types:       {summary["disfluency_types"]}')
        print(f'  Pitch CV:      {summary["avg_pitch_cv"]:.4f}')
        print(f'  Syllable Rate: {summary["avg_syllable_rate"]:.2f} syl/s')
        print(f'  Emotion:       {summary["dominant_emotion"]} (variety: {summary["emotion_variety"]:.2f})')
        if summary.get('emotion_distribution'):
            print(f'    Distribution: {summary["emotion_distribution"]}')
        print(f'  Timings:       {meta["timings"]}')

## 8. Copy outputs back to Drive

In [ ]:
# Save outputs to Google Drive for later use
drive_output = '/content/drive/MyDrive/voice_pipeline_outputs'
os.makedirs(drive_output, exist_ok=True)

for f in os.listdir('data/outputs'):
    if f.endswith('.json'):
        src = os.path.join('data/outputs', f)
        dst = os.path.join(drive_output, f)
        shutil.copy2(src, dst)
        print(f'Saved: {dst}')

print(f'\nOutputs saved to Drive: {drive_output}')